# PyTorch does the calculus
**Machine Learning · Week 2 (Regression and gradient descent) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/ml/w2d3.html

PyTorch gives you **tensors** (NumPy-like arrays) that remember how they were computed,
so it can find gradients for you (**autograd**). Then a 5-line loop trains anything.

In [ ]:
import numpy as np
import torch

torch.set_num_threads(1)          # tiny models run fastest on a single CPU thread

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])
print("a + b   =", a + b)
print("a * b   =", a * b)          # element by element
print("a @ b   =", a @ b)          # dot product
print("shape   =", a.shape, " dtype =", a.dtype)

## Autograd in one line
Mark a tensor with `requires_grad=True`, compute something from it, call `.backward()`,
and the derivative appears in `.grad`.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
loss = (3 * w - 1) ** 2
loss.backward()
print("loss          :", loss.item())
print("d loss / d w  :", w.grad.item())
print("by hand, 6(3w-1) at w = 2:", 6 * (3 * 2 - 1))

## The pizza loss again
Same data as Monday (feature scaled). PyTorch's gradients match the formulas from Wednesday.

In [ ]:
rng = np.random.default_rng(3)
x_cm = rng.uniform(20, 40, size=25)
price = 0.55 * x_cm - 2 + rng.normal(0, 1.4, size=25)
xs = (x_cm - x_cm.mean()) / x_cm.std()

X = torch.tensor(xs, dtype=torch.float32).reshape(-1, 1)      # 25 rows, 1 feature
Y = torch.tensor(price, dtype=torch.float32).reshape(-1, 1)

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
loss = ((w * X + b - Y) ** 2).mean()
loss.backward()
print("autograd:", round(w.grad.item(), 4), round(b.grad.item(), 4))
print("by hand :", round(2 * np.mean((0 - price) * xs), 4), round(2 * np.mean(0 - price), 4))

## The training loop
These five lines are the heartbeat of deep learning. `nn.Linear(1, 1)` is a line: it holds one weight and one bias.

In [ ]:
torch.manual_seed(0)
model = torch.nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

losses = []
for step in range(100):
    optimizer.zero_grad()                              # 1. forget old gradients
    prediction = model(X)                              # 2. forward pass
    loss = torch.nn.functional.mse_loss(prediction, Y) # 3. how wrong are we?
    loss.backward()                                    # 4. backward pass: gradients
    optimizer.step()                                   # 5. nudge every knob downhill
    losses.append(loss.item())

print(f"w = {model.weight.item():.3f}, b = {model.bias.item():.3f}, final MSE = {losses[-1]:.3f}")

**Try this:** delete `optimizer.zero_grad()` from the loop above and run it again.
PyTorch *adds* new gradients to old ones, so training goes wrong. That's why line 1 exists.

## Your turn: ten features at once
The diabetes dataset has 442 patients with 10 measurements each; the target is a disease-progression score.
A linear model with 10 inputs is `torch.nn.Linear(10, 1)`: 10 weights and 1 bias. The features are
standardized for you. Write the five-line training loop (500 steps) and compare with the exact answer.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

data = load_diabetes()
Xd = torch.tensor(data.data, dtype=torch.float32)
Xd = (Xd - Xd.mean(dim=0)) / Xd.std(dim=0)                        # standardize every column
Yd = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1)

torch.manual_seed(0)
model = torch.nn.Linear(10, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
# TODO: the five-line training loop, repeated 500 times

final = torch.nn.functional.mse_loss(model(Xd), Yd).item()
exact = LinearRegression().fit(Xd.numpy(), Yd.numpy())
best = np.mean((exact.predict(Xd.numpy()) - Yd.numpy()) ** 2)
print(f"PyTorch MSE: {final:.1f}   exact least squares: {best:.1f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
torch.manual_seed(0)
model = torch.nn.Linear(10, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
for step in range(500):
    optimizer.zero_grad()
    loss = torch.nn.functional.mse_loss(model(Xd), Yd)
    loss.backward()
    optimizer.step()

final = torch.nn.functional.mse_loss(model(Xd), Yd).item()
print(f"PyTorch MSE: {final:.1f}   exact least squares: {best:.1f}")

# with standardized features, the weights can be compared: which measurements matter most?
weights = model.weight.detach().numpy().ravel()
order = np.argsort(-np.abs(weights))
print("largest weights:", ", ".join(f"{data.feature_names[i]} {weights[i]:+.1f}" for i in order[:4]))
print("bmi = body-mass index, bp = blood pressure, s1..s6 = blood measurements")